# 03 — Búsqueda de hiperparámetros: XGBoost

Gradient boosting de árboles: modelo no lineal fuerte para datos tabulares. Tiene
muchos ejes de regularización, y los buscamos todos:

- Complejidad de cada árbol: `max_depth`, `min_child_weight`, `gamma` (poda).
- Regularización de las hojas: `reg_lambda` (L2), `reg_alpha` (L1).
- Randomización tipo bagging: `subsample`, `colsample_bytree`.
- Shrinkage vs. nº de árboles: `learning_rate`, `n_estimators`.

Como el espacio es enorme, usamos **búsqueda aleatoria** (`n_iter`) con la misma
CV temporal del notebook 02. El test no se toca hasta el final.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))          # componente_b/ (datos, evaluacion)
warnings.filterwarnings('ignore')                  # silenciar ConvergenceWarning de sklearn

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev
from modelos import (LinearRegressor, XGBoostRegressor, NeuralNetRegressor,
                     RandomForestRegressorModel, HistGBMRegressor, StackingRegressorModel)

# Cultivo del estudio (cambiar a 'maiz' para reproducir con maíz).
CULTIVO = 'soja'
ds = datos.prepare(CULTIVO, use_agro=True, enc_smooth=10.0)
print(f'{CULTIVO}: {len(ds.feature_cols)} features | '
      f'train {ds.X_train.shape[0]} filas (≤{datos.TRAIN_END}) | '
      f'test {ds.X_test.shape[0]} filas (≥{datos.TEST_START})')


## La búsqueda

Búsqueda aleatoria de 25 combinaciones sobre el grid, optimizando RMSE de validación temporal.

In [ ]:
grid = {
    'max_depth':        [3, 4, 6, 8],
    'learning_rate':    [0.02, 0.05, 0.1],
    'n_estimators':     [200, 400, 800],
    'subsample':        [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0],
    'min_child_weight': [1, 5],
    'reg_lambda':       [1.0, 5.0],
    'reg_alpha':        [0.0, 1.0],
    'gamma':            [0.0, 1.0],
}
tabla, best = ev.buscar(XGBoostRegressor, grid, ds, metric='rmse',
                        n_iter=25, random_state=42, n_splits=4)
print('Mejores hiperparámetros:', best)
tabla.head(10)

## Modelo final

Reentrenamos con los mejores hiperparámetros sobre todo el train y evaluamos en el
test. **Métricas del modelo final:**

In [ ]:
modelo = XGBoostRegressor(**best, random_state=42).fit(ds.X_train, ds.y_train)
pred = modelo.predict(ds.X_test)
ev.tabla([ev.evaluar('XGBoost (final)', pred, ds),
          ev.evaluar('baseline media x depto', ev.pred_media_depto(ds), ds)])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ev.plot_pred_vs_real(ds.y_test, pred, 'XGBoost (final)', color=ev.C_XGB, ax=axes[0])
ev.plot_residuos(ds.y_test, pred, 'Residuos', color=ev.C_XGB, ax=axes[1])
plt.tight_layout(); plt.show()

## Importancia de features

Qué variables usa más el modelo para partir los árboles.

In [ ]:
imp = pd.Series(modelo._model.feature_importances_, index=ds.feature_cols)
imp = imp.sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(imp.index[::-1], imp.values[::-1], color=ev.C_XGB)
ax.set_title('XGBoost — top 12 feature importances'); plt.tight_layout(); plt.show()

## El mejor modelo del Componente A como features

Con la **configuración final ya elegida**, evaluamos en test el dataset (único,
unificado: clima + NDVI + ERA5) contra tres estrategias que reusan el **mejor
detector del Componente A** (el VAE `recon_prob`, que aprendió a representar el
clima "normal"): concatenar su **espacio latente**, hacer la regresión **solo en
el latente**, y agregar la categórica **`es_anomalo`** (su score umbralado).

El latente/score se computan en `latente.py` (y se cachean). *La primera corrida
entrena el VAE, así que tarda unos minutos.*

In [ ]:
tabla_lat = ev.comparar_latente(
    XGBoostRegressor, best, CULTIVO, fixed={'random_state': 42},
    vae_kwargs=dict(score_seeds=(42, 43, 44)))
tabla_lat

## Conclusión

XGBoost captura interacciones no lineales entre clima, espacio y tendencia que el
lineal no puede. Ojo con una limitación de los árboles: **no extrapolan** la
tendencia temporal fuera del rango de train (el `year` de test, 2021–2024, está por
encima de todo lo visto), algo que el lineal sí hace. La comparación final (05) lo
pone en contexto contra el lineal y la red neuronal.